# 选做实验：诊断 Bias 与 Variance

上一个选做实验通过训练误差和 Cross-validation Error 评估学习算法，帮助你量化模型表现并选择模型。本实验将进一步利用这些误差，判断下一步如何改进模型。它们可以提示模型是否存在 High Bias（Underfitting，欠拟合）或 High Variance（Overfitting，过拟合），如下图所示：

<img src="images/C2_W3_BiasVariance.png" style="width:75%;" alt="High Bias、合适拟合与 High Variance 的比较">

最左图存在 High Bias：模型未能捕捉训练数据的规律，因此训练误差和 Cross-validation Error 都较高。最右图存在 High Variance：模型过度拟合 Training Set，虽然训练误差较低，但在新样本上表现较差，Cross-validation Error 较高。中间的模型较为理想，既能从 Training Set 学到规律，也能很好地泛化到未见过的数据。

针对 High Bias，可以尝试：

- 添加 Polynomial Features。
- 获取额外特征。
- 减小 Regularization Parameter。

针对 High Variance，可以尝试：

- 增大 Regularization Parameter。
- 减少特征数量。
- 获取更多训练样本。

本实验将逐一尝试这些方法。


## 建立 Baseline Performance

诊断 High Bias 或 High Variance 之前，先了解合理的误差水平通常很有帮助。课程中介绍了以下 Baseline Performance（基准表现）的来源：

- Human-level Performance。
- 其他算法的表现。
- 根据经验作出的估计。

真实数据可能包含大量噪声，因此通常无法达到 0% 的错误率。例如，一个 Computer Vision 应用的训练错误率为 10%，Cross-validation 错误率为 15%，你可能首先怀疑存在 High Bias。但随后发现，人类的错误率也无法低于 10%。若以此为 Baseline，就应优先解决 High Variance，缩小 Cross-validation Error 与训练误差之间的差距。

下面以此为基础，探索解决这些问题的方法。


## 导入软件包与设置实验环境

除 Scikit-learn 中的几个 [Linear Regressors](https://scikit-learn.org/stable/modules/classes.html#classical-linear-regressors) 外，本实验所用函数都位于当前目录的 `utils.py`。大部分代码与上一个实验相同，因此不再逐行展示。辅助函数主要用于划分数据，以及遍历 Polynomial Degree、Regularization Parameter 等参数，绘制对应的训练误差与 Cross-validation Error。可以打开该文件查看具体实现。


In [ ]:
# for building linear regression models
from sklearn.linear_model import LinearRegression, Ridge

# import lab utility functions in utils.py
import utils 

## 解决 High Bias

先讨论模型 Underfitting 时可以采取的方法：此时训练误差明显高于 Baseline Performance。


### 尝试添加 Polynomial Features

上个实验已经展示过这一方法。Polynomial Features 能帮助模型学习更复杂的数据规律。下面使用一个包含一个输入特征和一个目标的合成 Regression 数据集，观察增加多项式次数时训练误差与 Cross-validation Error 的变化。还将人为设定一个 Baseline，并在图中显示。


In [ ]:
# Split the dataset into train, cv, and test
x_train, y_train, x_cv, y_cv, x_test, y_test = utils.prepare_dataset('data/c2w3_lab2_data1.csv')

print(f"the shape of the training set (input) is: {x_train.shape}")
print(f"the shape of the training set (target) is: {y_train.shape}\n")
print(f"the shape of the cross validation set (input) is: {x_cv.shape}")
print(f"the shape of the cross validation set (target) is: {y_cv.shape}\n")

# Preview the first 5 rows
print(f"first 5 rows of the training inputs (1 feature):\n {x_train[:5]}\n")

# Instantiate the regression model class
model = LinearRegression()

# Train and plot polynomial regression models
utils.train_plot_poly(model, x_train, y_train, x_cv, y_cv, max_degree=10, baseline=400)

可以看到，添加的 Polynomial Features 越多，模型对 Training Set 的拟合越好。本例中，一些模型甚至优于设定的 Baseline。次数大于 4 的模型表现接近或优于 Baseline，因此可以认为它们的 Bias 较低。

但如果将 Baseline Error 设得更低，例如根据领域专家对可接受误差的意见调整标准，这些模型仍可能被视为 High Bias。此时可以继续尝试其他改进方法。


In [ ]:
# Train and plot polynomial regression models. Bias is defined lower.
utils.train_plot_poly(model, x_train, y_train, x_cv, y_cv, max_degree=10, baseline=250)

### 尝试获取额外特征

另一个方法是获取其他特征。假设得到上述结果后，你进行了新一轮数据采集，增加了一个特征。如下所示，数据集现在包含两列输入特征。


In [ ]:
x_train, y_train, x_cv, y_cv, x_test, y_test = utils.prepare_dataset('data/c2w3_lab2_data2.csv')

print(f"the shape of the training set (input) is: {x_train.shape}")
print(f"the shape of the training set (target) is: {y_train.shape}\n")
print(f"the shape of the cross validation set (input) is: {x_cv.shape}")
print(f"the shape of the cross validation set (target) is: {y_cv.shape}\n")

# Preview the first 5 rows
print(f"first 5 rows of the training inputs (2 features):\n {x_train[:5]}\n")

使用相同流程重新训练。可以看到，训练误差现在更接近 Baseline，甚至可能优于 Baseline。


In [ ]:
# Instantiate the model class
model = LinearRegression()

# Train and plot polynomial regression models. Dataset used has two features.
utils.train_plot_poly(model, x_train, y_train, x_cv, y_cv, max_degree=6, baseline=250)

### 尝试减小 Regularization Parameter

此时可能希望引入 Regularization 来防止 Overfitting，但 Regularization Parameter 过大会导致 Underfitting。下面使用 [`Ridge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html#sklearn.linear_model.Ridge) 训练四次 Polynomial Regression 模型。它允许设置 Regularization Parameter，即 $\lambda$。你将尝试多个值并比较结果。


In [ ]:
# Define lambdas to plot
reg_params = [10, 5, 2, 1, 0.5, 0.2, 0.1]

# Define degree of polynomial and train for each value of lambda
utils.train_plot_reg_params(reg_params, x_train, y_train, x_cv, y_cv, degree= 4, baseline=250)

图中初始的 $\lambda$ 为 `10`，此时训练误差高于 Baseline，说明对参数 `w` 的惩罚过强，阻碍了模型学习复杂规律。减小 $\lambda$ 后，这种限制逐渐放宽，训练误差可以接近 Baseline Performance。


## 解决 High Variance

接下来讨论模型对 Training Set 发生 Overfitting 时可以采取的方法。目标是提高模型对新样本的 Generalization 能力，因此重点是降低 Cross-validation Error。


### 尝试增大 Regularization Parameter

与上一部分相反，过小的 Regularization Parameter 虽然能使 Bias 保持较低，却可能无法有效改善 Variance。如下所示，增大 $\lambda$ 可以降低 Cross-validation Error。


In [ ]:
# Define lambdas to plot
reg_params = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1]

# Define degree of polynomial and train for each value of lambda
utils.train_plot_reg_params(reg_params, x_train, y_train, x_cv, y_cv, degree= 4, baseline=250)

### 尝试减少特征数量

上个实验已表明，过多的 Polynomial Terms 可能造成 Overfitting。可以减少这些项，寻找训练误差与 Cross-validation Error 之间较好的平衡。

数据包含无关特征时，减少特征也可能有帮助。例如，医院分配给患者的 ID 通常无助于诊断肿瘤，应从训练特征中去除。

下面对两个数据集分别进行 Polynomial Regression：一个是之前的两特征数据集，另一个在此基础上添加了一列随机 ID，包含三个特征。运行下方单元预览数据，可以看到前两列相同，第三列是新增的随机数。


In [ ]:
# Prepare dataset with randomID feature
x_train, y_train, x_cv, y_cv, x_test, y_test = utils.prepare_dataset('data/c2w3_lab2_data2.csv')

# Preview the first 5 rows
print(f"first 5 rows of the training set with 2 features:\n {x_train[:5]}\n")

# Prepare dataset with randomID feature
x_train, y_train, x_cv, y_cv, x_test, y_test = utils.prepare_dataset('data/c2w3_lab2_data3.csv')

# Preview the first 5 rows
print(f"first 5 rows of the training set with 3 features (1st column is a random ID):\n {x_train[:5]}\n")

训练模型并绘图。实线表示两个特征时的误差，虚线表示三个特征时的误差。可以看到，三个特征的数据集具有更高的 Cross-validation Error，尤其是在增加 Polynomial Terms 后，因为模型也试图从与目标无关的随机 ID 中学习规律。

再观察 `degree=4`：虽然三个特征时的训练误差更低，但训练误差与 Cross-validation Error 之间的差距明显更大。这也是 Overfitting 的信号。


In [ ]:
# Define the model
model = LinearRegression()

# Define properties of the 2 datasets
file1 = {'filename':'data/c2w3_lab2_data3.csv', 'label': '3 features', 'linestyle': 'dotted'}
file2 = {'filename':'data/c2w3_lab2_data2.csv', 'label': '2 features', 'linestyle': 'solid'}
files = [file1, file2]

# Train and plot for each dataset
utils.train_plot_diff_datasets(model, files, max_degree=4, baseline=250)

### 获取更多训练样本

最后，可以尝试增加样本数量以降低 Cross-validation Error。下面训练四次 Polynomial Regression 模型，并绘制 Learning Curve（学习曲线），观察误差如何随样本数量变化。


In [ ]:
# Prepare the dataset
x_train, y_train, x_cv, y_cv, x_test, y_test = utils.prepare_dataset('data/c2w3_lab2_data4.csv')
print(f"the shape of the entire training set (input) is: {x_train.shape}")
print(f"the shape of the entire training set (target) is: {y_train.shape}\n")
print(f"the shape of the entire cross validation set (input) is: {x_cv.shape}")
print(f"the shape of the entire cross validation set (target) is: {y_cv.shape}\n")

# Instantiate the model class
model = LinearRegression()

# Define the degree of polynomial and train the model using subsets of the dataset.
utils.train_plot_learning_curve(model, x_train, y_train, x_cv, y_cv, degree= 4, baseline=250)

随着数据集增大，Cross-validation Error 逐渐接近训练误差。图中还说明，增加样本通常无法解决 High Bias：即使数据量增大，训练误差仍然相对平稳。


## 实验回顾

本实验练习了应对学习算法 High Bias 和 High Variance 的方法。识别这些问题，有助于判断模型开发的下一步该尝试什么。接下来的课程将进一步探讨 Machine Learning 开发流程，以及实际项目中需要考虑的其他因素。
